# Tobroco Data Mining Project

## Project Context

This notebook presents a comprehensive data understanding analysis of **6.5 million time registration records** spanning **10,286 production orders** across **183 assembly zones**. The analysis reveals **critical data quality challenges** at step-level whilst confirming that **zone-level aggregation provides a more robust foundation** for predictive modelling.

**Key Finding**: **79% of all step-level records are under 5 seconds**, indicating widespread "click-through" behaviour where operators rapidly advance through assembly steps without accurate time recording.

In [1]:
# Import libraries
import pyodbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import sklearn
warnings.filterwarnings('ignore')

## 1. Data connection

In [2]:
# Database connection string
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=tobroco-ai-db.database.windows.net;"
    "Database=thingworx_data;"
    "UID=sqladmin;"
    "PWD=>Qy:NhMV[F1DX2h'!%;"
)

# Establish connection
conn = pyodbc.connect(conn_str)
print("✅ Connected to Azure SQL Database")

✅ Connected to Azure SQL Database


In [3]:
query = """
WITH tr AS (
    SELECT
        t.productionordernumber,
        COUNT(DISTINCT t.assemblystepid) AS amountOfSteps,
        SUM(DATEDIFF_BIG(second, t.startdate, t.enddate)) AS durationInSeconds
    FROM ninea_timeregistrations t
    WHERE t.assemblystepid IS NOT NULL
      AND t.startdate <> t.enddate
      AND t.productionordernumber IS NOT NULL
    GROUP BY t.productionordernumber
    HAVING SUM(DATEDIFF_BIG(second, t.startdate, t.enddate)) < 1209600 --filter 'bad' measurements
       AND SUM(DATEDIFF_BIG(second, t.startdate, t.enddate)) > 60
),
po_agg AS (
    SELECT
        po.productionordernumber,
        MIN(po.dossierdescription) AS machineName,
        COUNT(*) AS amountOfOperations,
        COUNT(DISTINCT po.zoneuid) AS amountOfZones,
        COUNT(DISTINCT a.articlenumber) AS amountOfOptions
    FROM ninea_productionoperation po
    JOIN ninea_assemblystep a ON a.assemblystepid = po.assemblystepid
    WHERE po.productionordernumber IS NOT NULL
      AND po.dossierdescription IS NOT NULL
      AND po.dossierdescription <> ''
      AND po.assemblystepid IS NOT NULL
    GROUP BY po.productionordernumber
)

SELECT po_agg.productionordernumber AS productionOrderNumber,
    po_agg.machineName,
    po_agg.amountOfOperations,
    po_agg.amountOfZones,
    tr.amountOfSteps,
    po_agg.amountOfOptions,
    tr.durationInSeconds / 3600.0 AS durationInHours
FROM po_agg
JOIN tr ON tr.productionordernumber = po_agg.productionordernumber
"""

df = pd.read_sql(query, conn)
df.head()

,productionOrderNumber,machineName,amountOfOperations,amountOfZones,amountOfSteps,amountOfOptions,durationInHours
0,PO00242569,Giant G1500 X-tra,210,27,184,11,4.649166
1,PO00242571,Giant G1500 X-tra,786,73,700,39,37.434444
2,PO00242576,Giant G1500,848,73,788,39,47.318611
3,PO00242577,Giant G1500 X-tra,765,77,684,31,34.134444
4,PO00291884,Giant G2500 X-tra HD,6,1,5,3,0.599166


In [4]:
#Delete outliers

Q1 = df["durationInHours"].quantile(0.25)
Q3 = df["durationInHours"].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

df_filtered = df[
    (df["durationInHours"] >= lower) &
    (df["durationInHours"] <= upper)
]

In [5]:
rows_before = len(df)
rows_after = len(df_filtered)
rows_skipped = rows_before - rows_after
skipped_pct = rows_skipped / rows_before * 100

print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Rows skipped: {rows_skipped}")
print(f"Skipped percentage: {skipped_pct:.2f}%")

Rows before: 3598
Rows after: 2805
Rows skipped: 793
Skipped percentage: 22.04%


## 2. Random Forest

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [7]:
df_filtered.columns.tolist()

['productionOrderNumber',
 'machineName',
 'amountOfOperations',
 'amountOfZones',
 'amountOfSteps',
 'amountOfOptions',
 'durationInHours']

In [8]:
#Split features and label
X = df_filtered.drop(columns=["durationInHours", "productionOrderNumber", "machineName"])
y = df_filtered["durationInHours"]

In [9]:
#Split train and test set
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [10]:
# Build model
model = RandomForestRegressor(
    n_estimators=100,
    random_state=45
)

In [11]:
# Train
model.fit(X_train, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [12]:
# Predict
y_pred = model.predict(X_test)

In [13]:
# Evaluate
print("MAE:", mean_absolute_error(y_test, y_pred))
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MAE: 4.288503758101159
MSE: 37.830149840313155
R2: 0.7902785374843306


#### This went from very bad (negative R2 with all outliers) to much better 66% (taking into account the outliers)